# Ironhack Lab | LoRA Tuning using PEFT

Run each section in order with **Shift+Enter**. Each step prints a result you can check. The first download requires internet access and space for two model checkpoints. The quick run is a small experiment and CPU training may still take several minutes.

LoRA learns a low-rank update to selected weight matrices while freezing the original weights. This solution changes the example from prompt continuation to passage-based question answering using SQuAD, with GPT-2 and SmolLM2 in place of BLOOM.

[Ironhack starter](https://github.com/ironhack-labs/lab-lora-tuning-peft/blob/main/lab-lora-tuning-peft.ipynb) · [SQuAD](https://huggingface.co/datasets/rajpurkar/squad) · [GPT-2](https://huggingface.co/openai-community/gpt2) · [SmolLM2](https://huggingface.co/HuggingFaceTB/SmolLM2-135M) · [PEFT LoRA docs](https://huggingface.co/docs/peft/main/en/package_reference/lora)


## 1. Check the Python kernel

Checks for the libraries this lab needs. If any are missing, install them in this kernel with the commented command, then restart the kernel and begin again.

In [1]:
# %pip install "torch>=2.6" "transformers==5.18.0" "peft==0.21.2" "datasets==5.0.1" "accelerate>=1,<2"
from importlib.metadata import PackageNotFoundError, version
import sys

packages = ("torch", "transformers", "peft", "datasets", "accelerate")
missing = []
for package in packages:
    try:
        print(f"{package}: {version(package)}")
    except PackageNotFoundError:
        missing.append(package)
print("Python:", sys.executable)
if missing:
    raise RuntimeError(f"Missing {', '.join(missing)}. See the install line above.")

torch: 2.14.1+cpu
transformers: 5.18.0
peft: 0.21.2
datasets: 5.0.1
accelerate: 1.15.0
Python: C:\Users\Windows\python\python.exe


## 2. Set up the comparison

Keep the dataset and training budget fixed. Compare GPT-2 at two LoRA ranks, then try SmolLM2 with rank 4.

In [2]:
import gc
import json
import re
import string
from collections import Counter
from pathlib import Path
from time import perf_counter

import torch
from datasets import Dataset, load_dataset
from IPython.display import Markdown, display
from peft import LoraConfig, PeftModel, TaskType, get_peft_model
from transformers import AutoModelForCausalLM, AutoTokenizer, Trainer, TrainingArguments, set_seed

SEED = 42
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if DEVICE.type == "cpu":
    torch.set_num_threads(min(torch.get_num_threads(), 4))
MAX_LENGTH = 256
TRAIN_ROWS, EVAL_ROWS = 40, 12
TRAIN_STEPS = 20
NEW_TOKENS = 16
OUTPUT_DIR = Path("peft_lab_outputs")
MODELS = {
    "gpt2": ("openai-community/gpt2", ["c_attn"]),
    "smollm2": ("HuggingFaceTB/SmolLM2-135M", ["q_proj", "v_proj"]),
}
print("Device:", DEVICE, "| steps per experiment:", TRAIN_STEPS)

Device: cpu | steps per experiment: 20


## 3. Prepare the SQuAD dataset

Build a passage and question prompt. Use the same short examples for both tokenizers, with training and evaluation drawn from separate official splits.

This machine uses local copies of the official SQuAD files and model checkpoints. If you run the notebook elsewhere without that folder, it downloads the same public resources from Hugging Face.

In [3]:
ASSETS_DIR = Path("lora_lab_assets")

def model_source(name):
    local_model = ASSETS_DIR / name
    if local_model.is_dir():
        return str(local_model), True
    return MODELS[name][0], False

tokenizers = {}
for name in MODELS:
    source, is_local = model_source(name)
    print(f"Loading {name} tokenizer from {'local files' if is_local else 'Hugging Face'}")
    try:
        tokenizers[name] = AutoTokenizer.from_pretrained(source, local_files_only=is_local)
    except OSError as error:
        raise RuntimeError(f"Could not load {name} from {source}. Fix this step before continuing.") from error
for tokenizer in tokenizers.values():
    tokenizer.pad_token = tokenizer.eos_token

def make_prompt(row):
    return f"Passage: {row['context'].strip()}\nQuestion: {row['question'].strip()}\nAnswer:"

def encode_example(row, tokenizer):
    prompt = tokenizer.encode(make_prompt(row), add_special_tokens=False)
    answer = tokenizer.encode(" " + row["answers"]["text"][0].strip(), add_special_tokens=False)
    ids = prompt + answer + [tokenizer.eos_token_id]
    return {"input_ids": ids, "labels": [-100] * len(prompt) + ids[len(prompt):]}

def take_short_rows(split, count):
    from random import Random

    from pyarrow.parquet import ParquetFile

    local_file = ASSETS_DIR / "squad" / "plain_text" / f"{split}-00000-of-00001.parquet"
    if local_file.is_file():
        def local_candidates():
            rng = Random(SEED)
            parquet = ParquetFile(local_file)
            for batch in parquet.iter_batches(
                batch_size=128, columns=["id", "context", "question", "answers"],
                use_threads=False,
            ):
                examples = batch.to_pylist()
                rng.shuffle(examples)
                yield from examples

        candidates = local_candidates()
    else:
        source = load_dataset("rajpurkar/squad", split=split, streaming=True)
        candidates = source.shuffle(seed=SEED, buffer_size=512).take(5000)

    rows = []
    for row in candidates:
        if row["answers"]["text"] and all(
            len(encode_example(row, tok)["input_ids"]) <= MAX_LENGTH
            for tok in tokenizers.values()
        ):
            rows.append(row)
            if len(rows) == count:
                return rows
    raise RuntimeError(f"Found only {len(rows)} usable {split} rows.")

train_rows = take_short_rows("train", TRAIN_ROWS)
eval_rows = take_short_rows("validation", EVAL_ROWS)
assert not ({row["id"] for row in train_rows} & {row["id"] for row in eval_rows})
print(f"{len(train_rows)} training rows and {len(eval_rows)} held-out rows")
print(make_prompt(train_rows[0]), train_rows[0]["answers"]["text"][0])

Loading gpt2 tokenizer from local files
Loading smollm2 tokenizer from local files
40 training rows and 12 held-out rows
Passage: This Main Building, and the library collection, was entirely destroyed by a fire in April 1879, and the school closed immediately and students were sent home. The university founder, Fr. Sorin and the president at the time, the Rev. William Corby, immediately planned for the rebuilding of the structure that had housed virtually the entire University. Construction was started on the 17th of May and by the incredible zeal of administrator and workers the building was completed before the fall semester of 1879. The library collection was also rebuilt and stayed housed in the new Main Building for years afterwards. Around the time of the fire, a music hall was opened. Eventually becoming known as Washington Hall, it hosted plays and musical acts put on by the school. By 1880, a science program was established at the university, and a Science Hall (today LaFortun

## 4. Define preprocessing and evaluation

Train on answer tokens only. For evaluation, generate an answer without revealing the reference and score normalized exact match and token F1 against the reference answers.

In [4]:
if "train_rows" not in globals() or "eval_rows" not in globals():
    raise RuntimeError("Complete Step 3 successfully before running Step 4.")

def collate_examples(features, tokenizer):
    width = max(len(row["input_ids"]) for row in features)
    inputs, masks, labels = [], [], []
    for row in features:
        length = len(row["input_ids"])
        pad = width - length
        inputs.append(row["input_ids"] + [tokenizer.pad_token_id] * pad)
        masks.append([1] * length + [0] * pad)
        labels.append(row["labels"] + [-100] * pad)
    return {key: torch.tensor(values) for key, values in {
        "input_ids": inputs, "attention_mask": masks, "labels": labels
    }.items()}

def normalize(answer):
    answer = answer.lower().translate(str.maketrans("", "", string.punctuation))
    return " ".join(re.sub(r"\b(a|an|the)\b", " ", answer).split())

def answer_scores(prediction, references):
    guess = normalize(prediction).split()
    exact, f1 = [], []
    for reference in references:
        gold = normalize(reference).split()
        overlap = sum((Counter(guess) & Counter(gold)).values())
        exact.append(float(guess == gold))
        f1.append(2 * overlap / (len(guess) + len(gold)) if overlap else float(guess == gold))
    return max(exact), max(f1)

@torch.inference_mode()
def evaluate(model, tokenizer):
    model.eval()
    rows = []
    for row in eval_rows:
        inputs = tokenizer(make_prompt(row), add_special_tokens=False, return_tensors="pt").to(DEVICE)
        generated = model.generate(
            **inputs, max_new_tokens=NEW_TOKENS, do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
        )
        tail = generated[0, inputs.input_ids.shape[1]:]
        lines = tokenizer.decode(tail, skip_special_tokens=True).strip().splitlines()
        prediction = lines[0] if lines else ""
        exact, f1 = answer_scores(prediction, row["answers"]["text"])
        rows.append({"question": row["question"], "gold": row["answers"]["text"],
                     "prediction": prediction, "exact": exact, "f1": f1})
    return {"exact": sum(r["exact"] for r in rows) / len(rows),
            "f1": sum(r["f1"] for r in rows) / len(rows), "examples": rows}

for tokenizer in tokenizers.values():
    example = encode_example(train_rows[0], tokenizer)
    batch = collate_examples([example], tokenizer)
    assert batch["input_ids"].shape == batch["labels"].shape
    assert -100 in example["labels"] and tokenizer.eos_token_id == example["labels"][-1]
print("Prompt masking and answer labels checked.")

Prompt masking and answer labels checked.


## 5. Measure the two base models

Run both pretrained models on the same held-out questions before tuning. Later cells compare against these scores and predictions.

In [5]:
def load_base(name):
    source, is_local = model_source(name)
    return AutoModelForCausalLM.from_pretrained(
        source, local_files_only=is_local, use_safetensors=True,
        dtype=torch.float32 if DEVICE.type == "cpu" else "auto"
    ).to(DEVICE)

def free_unused_memory():
    gc.collect()
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()

baseline = {}
for name in MODELS:
    model = load_base(name)
    baseline[name] = evaluate(model, tokenizers[name])
    print(f"{name}: exact match {baseline[name]['exact']:.1%}, F1 {baseline[name]['f1']:.1%}")
    del model
    free_unused_memory()

gpt2: exact match 0.0%, F1 11.1%


smollm2: exact match 0.0%, F1 6.6%


## 6. Define LoRA training

Attach trainable adapters to attention projections, train them, and save the adapter weights. GPT-2 uses its fused attention projection and SmolLM2 uses separate query and value projections.

Training processes one example at a time and accumulates gradients across two examples before each update. Gradient checkpointing saves memory by recomputing some activations during backpropagation.

In [6]:
def train_adapter(name, rank):
    set_seed(SEED)
    _, targets = MODELS[name]
    tokenizer = tokenizers[name]
    base = load_base(name)
    # Set the same next-token loss that Transformers otherwise selects as a fallback.
    base.loss_type = "ForCausalLM"
    base.config.use_cache = False
    # Limit SmolLM2's backward pass to its final four layers for this CPU lab.
    adapter_layers = {}
    if name == "smollm2":
        depth = base.config.num_hidden_layers
        adapter_layers = {
            "layers_to_transform": list(range(depth - 4, depth)),
            "layers_pattern": "layers",
        }
    config = LoraConfig(
        task_type=TaskType.CAUSAL_LM,
        r=rank,
        lora_alpha=2 * rank,
        target_modules=targets,
        fan_in_fan_out=(name == "gpt2"),
        lora_dropout=0.05,
        bias="none",
        **adapter_layers,
    )
    model = get_peft_model(base, config)
    model.print_trainable_parameters()
    trainable, total = model.get_nb_trainable_parameters()
    assert 0 < trainable < total
    folder = OUTPUT_DIR / f"{name}-r{rank}"
    train_data = Dataset.from_list([encode_example(row, tokenizer) for row in train_rows])
    args = TrainingArguments(
        output_dir=str(folder), per_device_train_batch_size=1,
        gradient_accumulation_steps=2,
        gradient_checkpointing=(name == "gpt2"),
        gradient_checkpointing_kwargs={"use_reentrant": False},
        max_steps=TRAIN_STEPS, learning_rate=5e-4, logging_steps=5, disable_tqdm=True,
        save_strategy="no", report_to=[], seed=SEED, use_cpu=(DEVICE.type == "cpu"),
        dataloader_pin_memory=(DEVICE.type == "cuda"),
    )
    trainer = Trainer(
        model=model, args=args, train_dataset=train_data,
        data_collator=lambda examples: collate_examples(examples, tokenizer),
    )
    started = perf_counter()
    training = trainer.train()
    seconds = perf_counter() - started
    model.save_pretrained(folder)
    score = evaluate(model, tokenizer)
    result = {
        "model": name, "rank": rank, "adapter": str(folder), "trainable": trainable,
        "trainable_fraction": trainable / total, "seconds": seconds,
        "train_loss": float(training.training_loss), **score,
    }
    print(f"{name} r={rank}: exact match {score['exact']:.1%}, F1 {score['f1']:.1%}")
    del trainer, model, base
    free_unused_memory()
    return result

print("Training function ready. The next three cells run the experiments.")

Training function ready. The next three cells run the experiments.


## 7a. Train GPT-2 at rank 4

Run the reference LoRA configuration.

In [7]:
gpt2_r4 = train_adapter('gpt2', 4)

trainable params: 147,456 || all params: 124,587,264 || trainable%: 0.1184


Step,Training Loss
5,5.144859
10,5.051442
15,4.095071
20,3.442590


gpt2 r=4: exact match 16.7%, F1 20.0%


## 7b. Train GPT-2 at rank 8

Change rank while keeping the data, steps, and alpha-to-rank ratio fixed.

In [8]:
gpt2_r8 = train_adapter('gpt2', 8)

trainable params: 294,912 || all params: 124,734,720 || trainable%: 0.2364


Step,Training Loss
5,5.068182
10,4.739479
15,3.568621
20,2.938513


gpt2 r=8: exact match 8.3%, F1 15.0%


## 7c. Train SmolLM2 at rank 4

Apply LoRA to query and value projections in the final four layers to limit CPU memory use. The GPT-2 runs adapt every layer.

In [9]:
smollm2_r4 = train_adapter('smollm2', 4)

Completed 20 training steps. Training time: 39.0s.
Trainable parameters: 30,720. Training loss: 2.985.
smollm2 r=4: exact match 0.0%, F1 6.6%


## 8. Compare results and reload an adapter

Compare each adapter with its own base model. Reload one saved adapter onto a fresh base model and verify the same held-out prediction.

In [10]:
results = [gpt2_r4, gpt2_r8, smollm2_r4]
lines = [
    "| Model | Rank | Base EM | Tuned EM | Base F1 | Tuned F1 | Trainable | Time |",
    "|---|---:|---:|---:|---:|---:|---:|---:|",
]
for run in results:
    base_score = baseline[run["model"]]
    lines.append(
        f"| {run['model']} | {run['rank']} | {base_score['exact']:.1%} | {run['exact']:.1%} | "
        f"{base_score['f1']:.1%} | {run['f1']:.1%} | "
        f"{run['trainable']:,} ({run['trainable_fraction']:.2%}) | {run['seconds']:.1f}s |"
    )
results_table = "\n".join(lines)
display(Markdown(results_table))

reloaded = PeftModel.from_pretrained(load_base("gpt2"), gpt2_r4["adapter"], is_trainable=False)
reloaded_score = evaluate(reloaded, tokenizers["gpt2"])
assert [r["prediction"] for r in reloaded_score["examples"]] == [
    r["prediction"] for r in gpt2_r4["examples"]
], "Reloaded predictions differ from the in-memory adapter."
del reloaded
free_unused_memory()
print("Saved adapter reproduced the held-out predictions.")

example = gpt2_r4["examples"][0]
print("Question:", example["question"])
print("Reference:", example["gold"])
print("Base:", repr(baseline["gpt2"]["examples"][0]["prediction"]))
print("Tuned:", repr(example["prediction"]))

| Model | Rank | Base EM | Tuned EM | Base F1 | Tuned F1 | Trainable | Time |
|---|---:|---:|---:|---:|---:|---:|---:|
| gpt2 | 4 | 0.0% | 16.7% | 11.1% | 20.0% | 147,456 (0.12%) | 83.0s |
| gpt2 | 8 | 0.0% | 8.3% | 11.1% | 15.0% | 294,912 (0.24%) | 84.0s |
| smollm2 | 4 | 0.0% | 0.0% | 6.6% | 6.6% | 30,720 (0.02%) | 39.0s |

Saved adapter reproduced the held-out predictions.
Question: What does AFC stand for?
Reference: ['American Football Conference', 'American Football Conference', 'American Football Conference']
Base: 'The AFC stands for the American Football Conference. The AFC is the American Football Conference'
Tuned: '"The AFC."'


## 9. Generate the one-page report

Create a short report from the actual run. Review the displayed example and add your own observation before submitting.

In [11]:
report_lines = [
    "# LoRA with PEFT on SQuAD",
    f"**Setup.** GPT-2 and SmolLM2 trained on {len(train_rows)} short SQuAD training examples "
    f"for {TRAIN_STEPS} steps. Each run used the same {len(eval_rows)} held-out questions.",
    results_table,
    "**What I learned.** LoRA trains a small set of parameters. "
    f"Changing GPT-2 rank from 4 to 8 changed the trainable count from {gpt2_r4['trainable']:,} "
    f"to {gpt2_r8['trainable']:,}, and token F1 from {gpt2_r4['f1']:.1%} to {gpt2_r8['f1']:.1%}.",
    "**Baseline comparison.** " + " ".join(
        f"{run['model']} rank {run['rank']}: F1 changed by "
        f"{100 * (run['f1'] - baseline[run['model']]['f1']):+.1f} percentage points."
        for run in results
    ),
    f"**Example.** {example['question']} Reference: {example['gold'][0]!r}. "
    f"GPT-2 base: {baseline['gpt2']['examples'][0]['prediction']!r}. "
    f"GPT-2 rank 4: {example['prediction']!r}.",
    "**Limitations.** The small sample and short training runs make these results exploratory. "
    "The models differ in architecture, tokenizer, targeted projections, and adapter layer coverage. SmolLM2 adapts its final four layers. "
    "The comparison across models cannot isolate the effect of any one of those factors.",
]
report = "\n\n".join(report_lines)
display(Markdown(report))
(OUTPUT_DIR / "lora_report.md").write_text(report + "\n", encoding="utf-8")
(OUTPUT_DIR / "results.json").write_text(
    json.dumps({"baseline": baseline, "experiments": results}, indent=2), encoding="utf-8"
)
print("Report and evaluation results saved in", OUTPUT_DIR.resolve())

# LoRA with PEFT on SQuAD

**Setup.** GPT-2 and SmolLM2 trained on 40 short SQuAD training examples for 20 steps. Each run used the same 12 held-out questions.

| Model | Rank | Base EM | Tuned EM | Base F1 | Tuned F1 | Trainable | Time |
|---|---:|---:|---:|---:|---:|---:|---:|
| gpt2 | 4 | 0.0% | 16.7% | 11.1% | 20.0% | 147,456 (0.12%) | 83.0s |
| gpt2 | 8 | 0.0% | 8.3% | 11.1% | 15.0% | 294,912 (0.24%) | 84.0s |
| smollm2 | 4 | 0.0% | 0.0% | 6.6% | 6.6% | 30,720 (0.02%) | 39.0s |

**What I learned.** LoRA trains a small set of parameters. Changing GPT-2 rank from 4 to 8 changed the trainable count from 147,456 to 294,912, and token F1 from 20.0% to 15.0%.

**Baseline comparison.** gpt2 rank 4: F1 changed by +8.9 percentage points. gpt2 rank 8: F1 changed by +3.9 percentage points. smollm2 rank 4: F1 changed by +0.0 percentage points.

**Example.** What does AFC stand for? Reference: 'American Football Conference'. GPT-2 base: 'The AFC stands for the American Football Conference. The AFC is the American Football Conference'. GPT-2 rank 4: '"The AFC."'.

**Limitations.** The small sample and short training runs make these results exploratory. The models differ in architecture, tokenizer, targeted projections, and adapter layer coverage. SmolLM2 adapts its final four layers. The comparison across models cannot isolate the effect of any one of those factors.

Report and evaluation results saved in peft_lab_outputs.


## Before submitting

Review the report and save the notebook with your outputs. Commit the notebook in your fork, open a pull request, and use the submission flow in the student portal. The generated adapter files stay local unless specifically requested.
